Dataset link: https://www.robots.ox.ac.uk/~vgg/data/pets/

Only cat images were extracted from the dataset for this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from collections import Counter
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

In [ ]:
images_folder = "/content/drive/MyDrive/images"

all_files = [f for f in os.listdir(images_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

# Step 2: Extract breed/label from filename (everything before the last underscore)
labels = [f.rsplit('_', 1)[0] for f in all_files]

# Step 3: Create a DataFrame
df = pd.DataFrame({
    'filename': all_files,
    'label': labels
})

# Step 4: Split into train, validation, test (e.g., 70% train, 15% val, 15% test)
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print(f"Total training images: {len(train_df)}")
print(f"Total validation images: {len(val_df)}")
print(f"Total test images: {len(test_df)}")

Total training images: 1680
Total validation images: 360
Total test images: 360


In [ ]:
# Step 5: ImageDataGenerators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

# Step 6: Flow from DataFrame
train_generator = train_datagen.flow_from_dataframe(
    train_df,
    directory=images_folder,
    x_col='filename',
    y_col='label',
    target_size=(224, 224),
    batch_size=64,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_dataframe(
    val_df,
    directory=images_folder,
    x_col='filename',
    y_col='label',
    target_size=(224, 224),
    batch_size=64,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_dataframe(
    test_df,
    directory=images_folder,
    x_col='filename',
    y_col='label',
    target_size=(224, 224),
    batch_size=64,
    class_mode='categorical',
    shuffle=False
)


Found 1680 validated image filenames belonging to 12 classes.
Found 360 validated image filenames belonging to 12 classes.
Found 360 validated image filenames belonging to 12 classes.


In [ ]:

# Function to count images per breed
def count_per_breed(df, set_name):
    breed_counts = df['label'].value_counts()
    print(f"\nNumber of images per breed in {set_name}:")
    for breed, count in breed_counts.items():
        print(f"{breed}: {count}")

# Count for training set
count_per_breed(train_df, "Training Set")

# Count for validation set
count_per_breed(val_df, "Validation Set")

# Count for test set
count_per_breed(test_df, "Test Set")



Number of images per breed in Training Set:
Maine_Coon: 140
Abyssinian: 140
Sphynx: 140
Bombay: 140
Birman: 140
British_Shorthair: 140
Ragdoll: 140
Persian: 140
Siamese: 140
Russian_Blue: 140
Bengal: 140
Egyptian_Mau: 140

Number of images per breed in Validation Set:
Bengal: 30
Bombay: 30
Egyptian_Mau: 30
Persian: 30
Russian_Blue: 30
Ragdoll: 30
Sphynx: 30
British_Shorthair: 30
Siamese: 30
Abyssinian: 30
Birman: 30
Maine_Coon: 30

Number of images per breed in Test Set:
Abyssinian: 30
Russian_Blue: 30
Persian: 30
Bengal: 30
Maine_Coon: 30
Egyptian_Mau: 30
Birman: 30
Ragdoll: 30
Sphynx: 30
Siamese: 30
British_Shorthair: 30
Bombay: 30


In [ ]:

num_classes = 12  # 12 cat breeds

# Step 1: Load DenseNet121 without the top classifier
base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224,224,3))

# Step 2: Freeze the base model initially
base_model.trainable = False

# Step 3: Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x)
output = Dense(num_classes, activation='softmax')(x)

# Step 4: Define the model
model = Model(inputs=base_model.input, outputs=output)

# Step 5: Compile the model
model.compile(optimizer=Adam(learning_rate=1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Optional: print summary
model.summary()


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_1… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 7,049,804 (26.89 MB)

 Trainable params: 12,300 (48.05 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

- Transfer Learning: A pre-trained model (e.g., DenseNet121 trained on ImageNet) is used as a fixed feature extractor. The base model is frozen, and only a new classification head is trained on the new dataset.
- Fine-Tuning: After initial training, some of the deeper layers of the pre-trained model are unfrozen and trained with a low learning rate. This allows the model to adapt high-level features to the new task.

Our pipeline below is Transfer Learning → then Fine-Tuning (on top of it)

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    steps_per_epoch=len(train_generator),
    validation_steps=len(val_generator)
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 580s 20s/step - accuracy: 0.1877 - loss: 2.4719 - val_accuracy: 0.5194 - val_loss: 1.6002
Epoch 2/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.4529 - loss: 1.6648 - val_accuracy: 0.7500 - val_loss: 1.1025
Epoch 3/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.6286 - loss: 1.2194 - val_accuracy: 0.8111 - val_loss: 0.8530
Epoch 4/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.7022 - loss: 1.0106 - val_accuracy: 0.8222 - val_loss: 0.7319
Epoch 5/5
27/27 ━━━━━━━━━━━━━━━━━━━━ 32s 1s/step - accuracy: 0.7512 - loss: 0.8558 - val_accuracy: 0.8528 - val_loss: 0.6380


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Step 1: Unfreeze the last 10 layers of the base model
for layer in base_model.layers[:-10]:
    layer.trainable = False
for layer in base_model.layers[-10:]:
    layer.trainable = True

# Step 2: Recompile the model with a smaller learning rate
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Step 3: Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,       # stop if val_loss doesn't improve for 5 epochs
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "PETSdensenet121_best_model.keras",  # save best model
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# Step 4: Fine-tune
history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    steps_per_epoch=len(train_generator),
    validation_steps=len(val_generator),
    callbacks=[early_stop, checkpoint]
)

Epoch 1/15
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7252 - loss: 0.8743
Epoch 1: val_loss improved from inf to 0.68130, saving model to PETSdensenet121_best_model.keras
27/27 ━━━━━━━━━━━━━━━━━━━━ 98s 3s/step - accuracy: 0.7256 - loss: 0.8729 - val_accuracy: 0.8278 - val_loss: 0.6813
Epoch 2/15
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7888 - loss: 0.7633
Epoch 2: val_loss improved from 0.68130 to 0.66613, saving model to PETSdensenet121_best_model.keras
27/27 ━━━━━━━━━━━━━━━━━━━━ 32s 1s/step - accuracy: 0.7884 - loss: 0.7627 - val_accuracy: 0.8222 - val_loss: 0.6661
Epoch 3/15
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8023 - loss: 0.6813   
Epoch 3: val_loss improved from 0.66613 to 0.62914, saving model to PETSdensenet121_best_model.keras
27/27 ━━━━━━━━━━━━━━━━━━━━ 31s 1s/step - accuracy: 0.8019 - loss: 0.6817 - val_accuracy: 0.8389 - val_loss: 0.6291
Epoch 4/15
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8142 - loss: 0.6677
Epoch 4: val_loss imp

In [ ]:
from tensorflow.keras.models import load_model

# Step 1: Load the best saved model
best_model = load_model("PETSdensenet121_best_model.keras")

In [ ]:
# Step 2: Evaluate on the test set
test_loss, test_acc = best_model.evaluate(test_generator)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc*100:.2f}%")

6/6 ━━━━━━━━━━━━━━━━━━━━ 91s 15s/step - accuracy: 0.8525 - loss: 0.5253
Test Loss: 0.5000
Test Accuracy: 85.00%


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Step 1: Get predictions from the model
y_pred_probs = best_model.predict(test_generator, steps=len(test_generator), verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# Step 2: True labels
y_true = test_generator.classes

# Step 3: Class labels mapping
class_labels = list(test_generator.class_indices.keys())

# Step 4: Overall classification report
report = classification_report(y_true, y_pred, target_names=class_labels)
print("Classification Report (per-class accuracy):\n")
print(report)

# Step 5: Optional: Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n", cm)

6/6 ━━━━━━━━━━━━━━━━━━━━ 28s 3s/step
Classification Report (per-class accuracy):

                   precision    recall  f1-score   support

       Abyssinian       0.80      0.80      0.80        30
           Bengal       0.96      0.73      0.83        30
           Birman       0.74      0.67      0.70        30
           Bombay       0.97      0.97      0.97        30
British_Shorthair       0.83      0.83      0.83        30
     Egyptian_Mau       0.90      0.87      0.88        30
       Maine_Coon       0.81      0.87      0.84        30
          Persian       0.97      0.93      0.95        30
          Ragdoll       0.63      0.80      0.71        30
     Russian_Blue       0.81      0.87      0.84        30
          Siamese       0.93      0.87      0.90        30
           Sphynx       0.94      1.00      0.97        30

         accuracy                           0.85       360
        macro avg       0.86      0.85      0.85       360
     weighted avg       0.86   